# Notebook 3 - Sentiment Analysis

training models to classify movie reviews as positive or negative using the IMDB dataset

In [ ]:
# imports
import os
os.makedirs('reports', exist_ok=True)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Download stopwords (only needed once)
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

print('libraries ready')

## Load Dataset

using 10000 samples to keep it fast

In [ ]:
# Load the IMDB dataset
imdb = pd.read_csv('data/IMDB Dataset.csv').sample(
    10000, random_state=42
).reset_index(drop=True)

print(f'Shape: {imdb.shape}')
print(imdb.head())
print('\nClass distribution:')
print(imdb['sentiment'].value_counts())

# Pie chart: Positive vs Negative
fig, ax = plt.subplots(figsize=(6, 6))
counts = imdb['sentiment'].value_counts()
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=['#4CAF50', '#F44336'], startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Sentiment Class Distribution', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('reports/sentiment_distribution.png', dpi=80)
plt.show()
plt.close('all')
print('Classes are balanced — no class imbalance issue!')

## Text Cleaning

removing html tags, punctuation and stopwords from reviews

In [ ]:
def clean_text(text):
    """
    Clean a raw review string:
    1. Convert to lowercase
    2. Remove HTML tags (e.g. <br />)
    3. Remove punctuation and digits
    4. Remove English stopwords
    Returns the cleaned string.
    """
    text = text.lower()                           # lowercase everything
    text = re.sub(r'<[^>]+>', ' ', text)          # remove HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)         # remove punctuation & numbers
    words = text.split()                           # split into words
    words = [w for w in words if w not in STOPWORDS]  # remove stopwords
    return ' '.join(words)


# Show before/after cleaning
sample_raw = imdb['review'].iloc[0]
sample_clean = clean_text(sample_raw)
print('Before cleaning:')
print(sample_raw[:300])
print('\nAfter cleaning:')
print(sample_clean[:300])

# apply to all reviews
print('\\nCleaning all 10,000 reviews...')
imdb['clean_review'] = imdb['review'].apply(clean_text)
print('done')

## Prepare Features

In [ ]:
# Convert labels: 'positive' -> 1, 'negative' -> 0
imdb['label'] = (imdb['sentiment'] == 'positive').astype(int)

X = imdb['clean_review']
y = imdb['label']

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training samples: {len(X_train):,}')
print(f'Test samples    : {len(X_test):,}')

# tfidf vectorizer, keeping top 5000 words
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_vec.shape}')

## Model A - Logistic Regression

In [ ]:
# train logistic regression
lr_model = LogisticRegression(max_iter=200, random_state=42)
lr_model.fit(X_train_vec, y_train)

lr_preds = lr_model.predict(X_test_vec)
lr_proba = lr_model.predict_proba(X_test_vec)[:, 1]

lr_accuracy = accuracy_score(y_test, lr_preds)
lr_auc      = roc_auc_score(y_test, lr_proba)

print(f'Logistic Regression Results:')
print(f'  Accuracy : {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)')
print(f'  ROC-AUC  : {lr_auc:.4f}')
print()
print(classification_report(y_test, lr_preds, target_names=['Negative', 'Positive']))

# confusion matrix
cm_lr = confusion_matrix(y_test, lr_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Logistic Regression — Confusion Matrix', fontsize=13)
plt.xlabel('Predicted', fontsize=11)
plt.ylabel('Actual', fontsize=11)
plt.tight_layout()
plt.savefig('reports/confusion_matrix_lr.png', dpi=80)
plt.show()
plt.close('all')

## Model B - Naive Bayes

In [ ]:
# Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

nb_preds = nb_model.predict(X_test_vec)
nb_proba = nb_model.predict_proba(X_test_vec)[:, 1]

nb_accuracy = accuracy_score(y_test, nb_preds)
nb_auc      = roc_auc_score(y_test, nb_proba)

print(f'Naive Bayes Results:')
print(f'  Accuracy : {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)')
print(f'  ROC-AUC  : {nb_auc:.4f}')
print()
print(classification_report(y_test, nb_preds, target_names=['Negative', 'Positive']))

# confusion matrix
cm_nb = confusion_matrix(y_test, nb_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Naive Bayes — Confusion Matrix', fontsize=13)
plt.xlabel('Predicted', fontsize=11)
plt.ylabel('Actual', fontsize=11)
plt.tight_layout()
plt.savefig('reports/confusion_matrix_nb.png', dpi=80)
plt.show()
plt.close('all')

## Model Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes'],
    'Accuracy': [lr_accuracy, nb_accuracy],
    'Precision': [
        precision_score(y_test, lr_preds),
        precision_score(y_test, nb_preds)
    ],
    'Recall': [
        recall_score(y_test, lr_preds),
        recall_score(y_test, nb_preds)
    ],
    'F1-Score': [
        f1_score(y_test, lr_preds),
        f1_score(y_test, nb_preds)
    ],
    'ROC-AUC': [lr_auc, nb_auc]
}).round(4)

print('=== Model Comparison ===')
print(comparison.to_string(index=False))
print('\nLogistic Regression is typically better on text classification tasks.')

## predict_sentiment() function

using logistic regression since it did better

In [ ]:
def predict_sentiment(review_text):
    """
    Predict sentiment of a raw movie review string.
    Returns 'Positive' or 'Negative' with confidence percentage.

    Args:
        review_text : str — raw review (can contain HTML, punctuation, etc.)

    Returns:
        tuple: (sentiment_label, confidence_pct)
    """
    # Step 1: Clean the raw text
    cleaned = clean_text(review_text)

    # Step 2: Convert to TF-IDF vector
    vec = tfidf.transform([cleaned])

    # Step 3: Predict class and probability
    prediction = lr_model.predict(vec)[0]
    probability = lr_model.predict_proba(vec)[0]

    label      = 'Positive' if prediction == 1 else 'Negative'
    confidence = probability[prediction] * 100

    return label, round(confidence, 2)


# Test with 3 sample reviews
test_reviews = [
    "This movie was absolutely brilliant! The acting was superb and the story was gripping.",
    "Terrible film. Boring plot, bad acting, complete waste of time.",
    "It was okay. Some parts were good but overall not very memorable."
]

print('=== Sentiment Predictions ===')
for review in test_reviews:
    label, conf = predict_sentiment(review)
    print(f'Review : {review[:70]}...')
    print(f'Result : {label} ({conf:.1f}% confidence)')
    print()

print('done')